In [1]:
# ============================================================================
# MSGR-RAG Component 3: Concept & Domain Mapping
# ============================================================================
# Purpose: Create multi-scale knowledge graph hierarchy with abstraction mappings
# Author: Pavan Kumar Reddy Yannam
# Date: December 2024
# ============================================================================


In [2]:
# # Component 3: Concept & Domain Mapping
# 
# **Objectives:**
# 1. Create concept-level abstractions (φ_EC: Entity → Concept)
# 2. Create domain-level abstractions (φ_CD: Concept → Domain)
# 3. Build abstraction mappings
# 4. Abstract entity relationships to concept level
# 5. Abstract concept relationships to domain level
# 6. Validate multi-scale hierarchy

In [3]:
# ## Section 1: Environment Setup

In [4]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

print("✓ Libraries imported successfully")

# Define paths
DATA_DIR = Path("../Datasets")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"

# Input files from Component 2
ENTITY_CATALOG_PATH = INTERMEDIATE_DIR / "entity_catalog.csv"
ENTITY_RELATIONSHIPS_PATH = INTERMEDIATE_DIR / "entity_relationships.csv"
SELECTED_DRUGS_PATH = INTERMEDIATE_DIR / "selected_drugs_100.csv"

# Output paths for this component
CONCEPT_MAPPINGS_PATH = INTERMEDIATE_DIR / "concept_mappings.csv"
DOMAIN_MAPPINGS_PATH = INTERMEDIATE_DIR / "domain_mappings.csv"
CONCEPT_RELATIONSHIPS_PATH = INTERMEDIATE_DIR / "concept_relationships.csv"
DOMAIN_RELATIONSHIPS_PATH = INTERMEDIATE_DIR / "domain_relationships.csv"
ABSTRACTION_REPORT_PATH = INTERMEDIATE_DIR / "abstraction_report.json"

ATC_CLASSIFICATION_PATH = INTERMEDIATE_DIR / "atc_domain_classification.csv"

print(f"✓ Input files:")
print(f"  - {ENTITY_CATALOG_PATH}")
print(f"  - {ENTITY_RELATIONSHIPS_PATH}")
print(f"  - {SELECTED_DRUGS_PATH}")
print(f"\n✓ Output files:")
print(f"  - {CONCEPT_MAPPINGS_PATH}")
print(f"  - {DOMAIN_MAPPINGS_PATH}")
print(f"  - {CONCEPT_RELATIONSHIPS_PATH}")
print(f"  - {DOMAIN_RELATIONSHIPS_PATH}")
print(f"  - {ABSTRACTION_REPORT_PATH}")


✓ Libraries imported successfully
✓ Input files:
  - ../Datasets/intermediate/entity_catalog.csv
  - ../Datasets/intermediate/entity_relationships.csv
  - ../Datasets/intermediate/selected_drugs_100.csv

✓ Output files:
  - ../Datasets/intermediate/concept_mappings.csv
  - ../Datasets/intermediate/domain_mappings.csv
  - ../Datasets/intermediate/concept_relationships.csv
  - ../Datasets/intermediate/domain_relationships.csv
  - ../Datasets/intermediate/abstraction_report.json


In [5]:
# ## Section 2: Load Data

In [6]:
print("Loading data from Component 2...")
print("=" * 80)

# Load entity catalog
entity_catalog_df = pd.read_csv(ENTITY_CATALOG_PATH)
print(f"✓ Entity catalog: {len(entity_catalog_df)} entities")

# Load entity relationships
entity_relationships_df = pd.read_csv(ENTITY_RELATIONSHIPS_PATH)
print(f"✓ Entity relationships: {len(entity_relationships_df):,} relationships")

# Load selected drugs info (for therapeutic area mapping)
selected_drugs_df = pd.read_csv(SELECTED_DRUGS_PATH)
print(f"✓ Selected drugs: {len(selected_drugs_df)} drugs")

# Preview
print("\nEntity catalog preview:")
print(entity_catalog_df.head(10))

print("\nEntity types distribution:")
print(entity_catalog_df['entity_type'].value_counts())

atc_classification_df = pd.read_csv(ATC_CLASSIFICATION_PATH)
print(f"✓ ATC classification loaded: {len(atc_classification_df)} drugs")
print(atc_classification_df['mapping_source'].value_counts())

# Map entity_name -> ATC-grounded domain, ONLY for the 74 unambiguous single-match drugs
atc_grounded_only = atc_classification_df[atc_classification_df['mapping_source'] == 'atc_grounded_single']
atc_domain_lookup = dict(zip(atc_grounded_only['name'], atc_grounded_only['atc_l1_domain']))

print(f"\nATC-grounded domain lookup built for {len(atc_domain_lookup)} drugs")
print(f"Sample: {dict(list(atc_domain_lookup.items())[:5])}")

Loading data from Component 2...
✓ Entity catalog: 149 entities
✓ Entity relationships: 103,552 relationships
✓ Selected drugs: 100 drugs

Entity catalog preview:
       entity_id         entity_name entity_type                  source
0  CONDITION_112                COPD   Condition  Therapeutic_Extraction
1  CONDITION_113           Infection   Condition  Therapeutic_Extraction
2  CONDITION_114               Cough   Condition  Therapeutic_Extraction
3  CONDITION_115              Angina   Condition  Therapeutic_Extraction
4  CONDITION_116      Hypothyroidism   Condition  Therapeutic_Extraction
5  CONDITION_117            Lymphoma   Condition  Therapeutic_Extraction
6  CONDITION_118   Gastric_Disorders   Condition  Therapeutic_Extraction
7  CONDITION_119          Thrombosis   Condition  Therapeutic_Extraction
8  CONDITION_120            Diabetes   Condition  Therapeutic_Extraction
9  CONDITION_121  Cardiac_Conditions   Condition  Therapeutic_Extraction

Entity types distribution:
entity

In [7]:
# ## Section 3: Define Concept-Level Mappings (φ_EC)
# 
# Map entities to their conceptual categories


In [8]:
print("Creating Concept-Level Mappings (φ_EC: Entity → Concept):")
print("=" * 80)

# ============================================================================
# 3.1: Drug → Drug_Class Mapping
# ============================================================================
print("\n1. Drug → Drug_Class Mapping")
print("-" * 80)

# Define drug class mappings based on therapeutic use and chemical structure
# These are common drug classes in pharmacology

DRUG_CLASS_MAPPINGS = {
    # NSAIDs (Non-Steroidal Anti-Inflammatory Drugs)
    'NSAIDs': [
        'Acetylsalicylic acid', 'Ibuprofen', 'Naproxen', 'Ketoprofen', 
        'Diclofenac', 'Indomethacin', 'Ketorolac', 'Piroxicam',
        'Meloxicam', 'Celecoxib', 'Diflunisal', 'Sulindac',
        'Etodolac', 'Nabumetone', 'Oxaprozin', 'Phenylbutazone'
    ],
    
    # Anticoagulants
    'Anticoagulants': [
        'Warfarin', 'Heparin', 'Enoxaparin', 'Lepirudin', 'Bivalirudin',
        'Dabigatran', 'Rivaroxaban', 'Apixaban', 'Edoxaban', 'Phenprocoumon'
    ],
    
    # Anticonvulsants
    'Anticonvulsants': [
        'Phenytoin', 'Carbamazepine', 'Valproic acid', 'Lamotrigine',
        'Levetiracetam', 'Topiramate', 'Gabapentin', 'Pregabalin',
        'Zonisamide', 'Lacosamide', 'Perampanel', 'Eslicarbazepine acetate',
        'Oxcarbazepine', 'Tiagabine', 'Vigabatrin', 'Stiripentol'
    ],
    
    # Antipsychotics
    'Antipsychotics': [
        'Haloperidol', 'Chlorpromazine', 'Fluphenazine', 'Risperidone',
        'Olanzapine', 'Quetiapine', 'Aripiprazole', 'Ziprasidone',
        'Clozapine', 'Paliperidone', 'Asenapine', 'Lurasidone',
        'Flupentixol', 'Zuclopenthixol', 'Pimozide', 'Promethazine',
        'Methotrimeprazine', 'Droperidol', 'Prochlorperazine'
    ],
    
    # Antibiotics
    'Antibiotics': [
        'Penicillin', 'Amoxicillin', 'Ampicillin', 'Ceftriaxone',
        'Cefazolin', 'Ciprofloxacin', 'Levofloxacin', 'Azithromycin',
        'Clarithromycin', 'Doxycycline', 'Tetracycline', 'Vancomycin',
        'Gentamicin', 'Tobramycin', 'Metronidazole', 'Clindamycin',
        'Rifampicin', 'Rifampin', 'Ceftezole', 'Telithromycin'
    ],
    
    # Chemotherapy Agents
    'Chemotherapy_Agents': [
        'Methotrexate', 'Cyclophosphamide', 'Doxorubicin', 'Cisplatin',
        'Carboplatin', 'Paclitaxel', 'Docetaxel', 'Vincristine',
        'Vinblastine', 'Etoposide', 'Irinotecan', 'Fluorouracil',
        'Ifosfamide', 'Azacitidine', 'Thiotepa', 'Busulfan'
    ],
    
    # Opioids
    'Opioids': [
        'Morphine', 'Codeine', 'Hydrocodone', 'Oxycodone',
        'Hydromorphone', 'Fentanyl', 'Tramadol', 'Methadone',
        'Buprenorphine', 'Tapentadol', 'Hydrocodone'
    ],
    
    # Benzodiazepines
    'Benzodiazepines': [
        'Diazepam', 'Lorazepam', 'Alprazolam', 'Clonazepam',
        'Midazolam', 'Temazepam', 'Triazolam', 'Chlordiazepoxide',
        'Oxazepam', 'Flurazepam'
    ],
    
    # Antidiabetic Agents
    'Antidiabetic_Agents': [
        'Metformin', 'Insulin', 'Glipizide', 'Glyburide',
        'Glimepiride', 'Pioglitazone', 'Rosiglitazone', 'Sitagliptin',
        'Saxagliptin', 'Linagliptin', 'Exenatide', 'Liraglutide'
    ],
    
    # Antihypertensives
    'Antihypertensives': [
        'Lisinopril', 'Enalapril', 'Ramipril', 'Losartan',
        'Valsartan', 'Amlodipine', 'Nifedipine', 'Diltiazem',
        'Verapamil', 'Metoprolol', 'Atenolol', 'Carvedilol',
        'Hydrochlorothiazide', 'Furosemide', 'Spironolactone',
        'Methyldopa', 'Hydralazine', 'Clonidine', 'Telmisartan'
    ],
    
    # Proton Pump Inhibitors
    'Proton_Pump_Inhibitors': [
        'Omeprazole', 'Esomeprazole', 'Lansoprazole', 'Pantoprazole',
        'Rabeprazole', 'Dexlansoprazole'
    ],
    
    # H2 Receptor Antagonists
    'H2_Receptor_Antagonists': [
        'Ranitidine', 'Famotidine', 'Cimetidine', 'Nizatidine'
    ],
    
    # Antiarrhythmics
    'Antiarrhythmics': [
        'Amiodarone', 'Sotalol', 'Flecainide', 'Propafenone',
        'Quinidine', 'Procainamide', 'Disopyramide', 'Dofetilide'
    ],
    
    # Immunosuppressants
    'Immunosuppressants': [
        'Cyclosporine', 'Tacrolimus', 'Sirolimus', 'Everolimus',
        'Mycophenolate', 'Azathioprine', 'Cyclophosphamide'
    ],
    
    # Corticosteroids
    'Corticosteroids': [
        'Prednisone', 'Prednisolone', 'Methylprednisolone', 'Dexamethasone',
        'Hydrocortisone', 'Betamethasone', 'Triamcinolone', 'Budesonide',
        'Fluticasone', 'Beclomethasone'
    ],
    
    # Antihistamines
    'Antihistamines': [
        'Diphenhydramine', 'Cetirizine', 'Loratadine', 'Fexofenadine',
        'Desloratadine', 'Hydroxyzine', 'Promethazine', 'Chlorpheniramine'
    ],
    
    # Bronchodilators
    'Bronchodilators': [
        'Albuterol', 'Salbutamol', 'Salmeterol', 'Formoterol',
        'Tiotropium', 'Ipratropium', 'Theophylline'
    ],
    
    # Antiemetics
    'Antiemetics': [
        'Ondansetron', 'Granisetron', 'Metoclopramide', 'Prochlorperazine',
        'Promethazine', 'Domperidone', 'Aprepitant'
    ],
    
    # Laxatives
    'Laxatives': [
        'Bisacodyl', 'Senna', 'Docusate', 'Lactulose',
        'Polyethylene glycol', 'Magnesium hydroxide', 'Lubiprostone',
        'Linaclotide', 'Methylnaltrexone'
    ],
    
    # GI Motility Agents
    'GI_Motility_Agents': [
        'Misoprostol', 'Lubiprostone', 'Linaclotide', 'Plecanatide'
    ],
    
    # Vitamin D Analogs
    'Vitamin_D_Analogs': [
        'Alfacalcidol', 'Calcitriol', 'Cholecalciferol', 'Ergocalciferol',
        'Paricalcitol', 'Doxercalciferol'
    ],
    
    # Anticholinergics
    'Anticholinergics': [
        'Atropine', 'Scopolamine', 'Glycopyrrolate', 'Glycopyrronium',
        'Ipratropium', 'Tiotropium', 'Benztropine', 'Trihexyphenidyl'
    ],
    
    # Muscle Relaxants
    'Muscle_Relaxants': [
        'Baclofen', 'Cyclobenzaprine', 'Methocarbamol', 'Tizanidine',
        'Carisoprodol', 'Metaxalone', 'Orphenadrine', 'Chlorzoxazone'
    ],
    
    # Iron Supplements
    'Iron_Supplements': [
        'Ferrous sulfate', 'Ferrous gluconate', 'Ferrous fumarate',
        'Iron', 'Iron dextran', 'Iron sucrose'
    ],
    
    # Other (catch-all for drugs not in above categories)
    'Other_Drugs': []
}

# Create reverse mapping (drug_name → drug_class)
drug_to_class = {}

for drug_class, drug_list in DRUG_CLASS_MAPPINGS.items():
    for drug in drug_list:
        drug_to_class[drug.lower()] = drug_class

# Apply mapping to drugs in entity catalog
drugs_df = entity_catalog_df[entity_catalog_df['entity_type'] == 'Drug'].copy()

def assign_domain(drug_name, existing_domain_mapping_logic_result):
    """
    Domain assignment with ATC-grounding as primary source.
    Priority: ATC L1 (unambiguous single match) > existing deterministic fallback.
    """
    if drug_name in atc_domain_lookup:
        return atc_domain_lookup[drug_name], 'atc_grounded'
    else:
        return existing_domain_mapping_logic_result, 'deterministic_fallback'

def map_drug_to_class(drug_name):
    """Map drug name to drug class"""
    drug_lower = drug_name.lower()
    
    # Check direct mapping
    if drug_lower in drug_to_class:
        return drug_to_class[drug_lower]
    
    # Check partial matches (for multi-word drugs)
    for mapped_drug, drug_class in drug_to_class.items():
        if mapped_drug in drug_lower or drug_lower in mapped_drug:
            return drug_class
    
    # Default to Other_Drugs
    return 'Other_Drugs'

drugs_df['drug_class'] = drugs_df['entity_name'].apply(map_drug_to_class)

print(f"✓ Mapped {len(drugs_df)} drugs to drug classes")
print(f"\nDrug class distribution:")
print(drugs_df['drug_class'].value_counts())

# ============================================================================
# 3.2: Condition → Disease_Category Mapping
# ============================================================================
print("\n\n2. Condition → Disease_Category Mapping")
print("-" * 80)

CONDITION_CATEGORY_MAPPINGS = {
    'Cardiovascular_Diseases': [
        'Hypertension', 'Heart_Failure', 'Cardiac_Conditions', 'Arrhythmia',
        'Angina', 'Myocardial_Infarction', 'Stroke', 'Thrombosis'
    ],
    
    'Metabolic_Disorders': [
        'Diabetes', 'Hyperglycemia', 'Hypoglycemia', 'Thyroid_Disorders',
        'Hypothyroidism', 'Hyperthyroidism'
    ],
    
    'Pain_Disorders': [
        'Pain', 'Migraine', 'Headache', 'Arthritis', 'Rheumatoid_Arthritis'
    ],
    
    'Inflammatory_Disorders': [
        'Inflammation', 'Arthritis', 'Rheumatoid_Arthritis'
    ],
    
    'Infectious_Diseases': [
        'Infection', 'Bacterial_Infection', 'Viral_Infection', 'Fungal_Infection'
    ],
    
    'Neurological_Disorders': [
        'Seizure', 'Epilepsy', 'Parkinsons_Disease', 'Alzheimers_Disease',
        'Migraine', 'Headache'
    ],
    
    'Psychiatric_Disorders': [
        'Depression', 'Anxiety', 'Psychosis', 'Schizophrenia'
    ],
    
    'Respiratory_Disorders': [
        'Asthma', 'Bronchospasm', 'COPD', 'Cough'
    ],
    
    'Gastrointestinal_Disorders': [
        'Ulcer', 'Gastric_Disorders', 'Nausea', 'Diarrhea', 'Constipation'
    ],
    
    'Oncological_Disorders': [
        'Cancer', 'Tumor', 'Neoplasm', 'Leukemia', 'Lymphoma'
    ],
    
    'Fever': [
        'Fever'
    ]
}

# Create reverse mapping
condition_to_category = {}

for category, condition_list in CONDITION_CATEGORY_MAPPINGS.items():
    for condition in condition_list:
        condition_to_category[condition.lower()] = category

# Apply mapping to conditions
conditions_df = entity_catalog_df[entity_catalog_df['entity_type'] == 'Condition'].copy()

def map_condition_to_category(condition_name):
    """Map condition to disease category"""
    condition_lower = condition_name.lower()
    
    if condition_lower in condition_to_category:
        return condition_to_category[condition_lower]
    
    # Partial match
    for mapped_condition, category in condition_to_category.items():
        if mapped_condition in condition_lower or condition_lower in mapped_condition:
            return category
    
    return 'Other_Conditions'

conditions_df['disease_category'] = conditions_df['entity_name'].apply(map_condition_to_category)

print(f"✓ Mapped {len(conditions_df)} conditions to disease categories")
print(f"\nDisease category distribution:")
print(conditions_df['disease_category'].value_counts())

# ============================================================================
# 3.3: Adverse_Effect → Toxicity_Category Mapping
# ============================================================================
print("\n\n3. Adverse_Effect → Toxicity_Category Mapping")
print("-" * 80)

TOXICITY_CATEGORY_MAPPINGS = {
    'Organ_Toxicity': [
        'Hepatotoxicity', 'Nephrotoxicity', 'Cardiotoxicity', 
        'Neurotoxicity', 'Ototoxicity'
    ],
    
    'Reproductive_Toxicity': [
        'Teratogenicity', 'Carcinogenicity'
    ],
    
    'Gastrointestinal_Effects': [
        'Nausea', 'Vomiting', 'Diarrhea', 'Constipation'
    ],
    
    'Cardiovascular_Effects': [
        'Hypotension', 'Hypertension', 'Bleeding'
    ],
    
    'Neurological_Effects': [
        'Drowsiness', 'Sedation', 'Dizziness'
    ],
    
    'Carcinogenic_Effects': [
        'Carcinogenicity'
    ]
}

# Create reverse mapping
effect_to_category = {}

for category, effect_list in TOXICITY_CATEGORY_MAPPINGS.items():
    for effect in effect_list:
        effect_to_category[effect.lower()] = category

# Apply mapping to adverse effects
effects_df = entity_catalog_df[entity_catalog_df['entity_type'] == 'Adverse_Effect'].copy()

def map_effect_to_category(effect_name):
    """Map adverse effect to toxicity category"""
    effect_lower = effect_name.lower()
    
    if effect_lower in effect_to_category:
        return effect_to_category[effect_lower]
    
    # Partial match
    for mapped_effect, category in effect_to_category.items():
        if mapped_effect in effect_lower or effect_lower in mapped_effect:
            return category
    
    return 'Other_Toxicity'

effects_df['toxicity_category'] = effects_df['entity_name'].apply(map_effect_to_category)

print(f"✓ Mapped {len(effects_df)} adverse effects to toxicity categories")
print(f"\nToxicity category distribution:")
print(effects_df['toxicity_category'].value_counts())

Creating Concept-Level Mappings (φ_EC: Entity → Concept):

1. Drug → Drug_Class Mapping
--------------------------------------------------------------------------------
✓ Mapped 95 drugs to drug classes

Drug class distribution:
drug_class
Other_Drugs                53
Anticonvulsants             4
NSAIDs                      4
Antibiotics                 4
Antipsychotics              3
Anticoagulants              3
Immunosuppressants          3
Opioids                     2
H2_Receptor_Antagonists     2
Antihistamines              2
Benzodiazepines             2
GI_Motility_Agents          2
Antiemetics                 2
Antihypertensives           2
Antidiabetic_Agents         1
Anticholinergics            1
Muscle_Relaxants            1
Vitamin_D_Analogs           1
Iron_Supplements            1
Laxatives                   1
Chemotherapy_Agents         1
Name: count, dtype: int64


2. Condition → Disease_Category Mapping
--------------------------------------------------------------

In [9]:
# ## Section 4: Create Concept Mappings DataFrame

In [10]:
print("Creating Concept Mappings DataFrame (φ_EC):")
print("=" * 80)

# Combine all entity-to-concept mappings
concept_mappings = []

# Drugs → Drug Classes
for idx, row in drugs_df.iterrows():
    concept_mappings.append({
        'entity_id': row['entity_id'],
        'entity_name': row['entity_name'],
        'entity_type': row['entity_type'],
        'concept_name': row['drug_class'],
        'concept_type': 'Drug_Class',
        'mapping_function': 'φ_EC'
    })

# Conditions → Disease Categories
for idx, row in conditions_df.iterrows():
    concept_mappings.append({
        'entity_id': row['entity_id'],
        'entity_name': row['entity_name'],
        'entity_type': row['entity_type'],
        'concept_name': row['disease_category'],
        'concept_type': 'Disease_Category',
        'mapping_function': 'φ_EC'
    })

# Adverse Effects → Toxicity Categories
for idx, row in effects_df.iterrows():
    concept_mappings.append({
        'entity_id': row['entity_id'],
        'entity_name': row['entity_name'],
        'entity_type': row['entity_type'],
        'concept_name': row['toxicity_category'],
        'concept_type': 'Toxicity_Category',
        'mapping_function': 'φ_EC'
    })

concept_mappings_df = pd.DataFrame(concept_mappings)

print(f"✓ Created {len(concept_mappings_df)} entity→concept mappings")
print(f"\nConcept type distribution:")
print(concept_mappings_df['concept_type'].value_counts())

print(f"\nSample mappings:")
print(concept_mappings_df.head(15))


Creating Concept Mappings DataFrame (φ_EC):
✓ Created 149 entity→concept mappings

Concept type distribution:
concept_type
Drug_Class           95
Disease_Category     42
Toxicity_Category    12
Name: count, dtype: int64

Sample mappings:
   entity_id        entity_name entity_type         concept_name concept_type  \
0     DRUG_0          Melphalan        Drug          Other_Drugs   Drug_Class   
1     DRUG_1          Exenatide        Drug  Antidiabetic_Agents   Drug_Class   
2    DRUG_10      Buprenorphine        Drug              Opioids   Drug_Class   
3    DRUG_11          Tiagabine        Drug      Anticonvulsants   Drug_Class   
4    DRUG_12       Perphenazine        Drug          Other_Drugs   Drug_Class   
5    DRUG_13     Glycopyrronium        Drug     Anticholinergics   Drug_Class   
6    DRUG_14          Rifabutin        Drug          Other_Drugs   Drug_Class   
7    DRUG_15          Oxaprozin        Drug               NSAIDs   Drug_Class   
8    DRUG_16         Diflunisal 

In [11]:
print("Building ATC-Grounded Entity→Domain Override:")
print("=" * 80)

ATC_L1_TO_DOMAIN = {
    "ALIMENTARY TRACT AND METABOLISM": "Gastroenterology",
    "BLOOD AND BLOOD FORMING ORGANS": "Hematology",
    "CARDIOVASCULAR SYSTEM": "Cardiology",
    "DERMATOLOGICALS": "General_Medicine",
    "GENITO URINARY SYSTEM AND SEX HORMONES": "Endocrinology",
    "SYSTEMIC HORMONAL PREPARATIONS, EXCL. SEX HORMONES AND INSULINS": "Endocrinology",
    "ANTIINFECTIVES FOR SYSTEMIC USE": "Infectious_Disease",
    "ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS": "Oncology",
    "MUSCULO-SKELETAL SYSTEM": "Rheumatology",
    "NERVOUS SYSTEM": "Neurology",
    "ANTIPARASITIC PRODUCTS, INSECTICIDES AND REPELLENTS": "Infectious_Disease",
    "RESPIRATORY SYSTEM": "Pulmonology",
    "SENSORY ORGANS": "General_Medicine",
    "VARIOUS": "General_Medicine",
}

atc_classification_df = pd.read_csv(INTERMEDIATE_DIR / "atc_domain_classification.csv")
atc_grounded_only = atc_classification_df[atc_classification_df['mapping_source'] == 'atc_grounded_single'].copy()

# Translate ATC L1 name -> existing domain vocabulary
atc_grounded_only['domain_name'] = atc_grounded_only['atc_l1_domain'].map(ATC_L1_TO_DOMAIN)

entity_to_domain_direct = dict(zip(atc_grounded_only['name'], atc_grounded_only['domain_name']))

print(f"✓ ATC-grounded overrides: {len(entity_to_domain_direct)} drugs")
print(f"Sample: {dict(list(entity_to_domain_direct.items())[:5])}")

ATC_ENTITY_DOMAIN_PATH = INTERMEDIATE_DIR / "atc_entity_domain_overrides.csv"
pd.DataFrame(list(entity_to_domain_direct.items()), columns=['entity_name', 'domain_name']).to_csv(
    ATC_ENTITY_DOMAIN_PATH, index=False
)
print(f"✓ Saved: {ATC_ENTITY_DOMAIN_PATH}")

Building ATC-Grounded Entity→Domain Override:
✓ ATC-grounded overrides: 74 drugs
Sample: {'Lepirudin': 'Hematology', 'Etacrynic acid': 'Cardiology', 'Famotidine': 'Gastroenterology', 'Phenprocoumon': 'Hematology', 'Telmisartan': 'Cardiology'}
✓ Saved: ../Datasets/intermediate/atc_entity_domain_overrides.csv


In [12]:
# ## Section 5: Define Domain-Level Mappings (φ_CD)
# 
# Map concepts to medical domains/specialties

In [13]:
print("Creating Domain-Level Mappings (φ_CD: Concept → Domain):")
print("=" * 80)

# ============================================================================
# Drug Classes → Medical Domains
# ============================================================================
print("\n1. Drug_Class → Medical_Domain Mapping")
print("-" * 80)

DRUG_CLASS_TO_DOMAIN = {
    # Cardiology
    'Cardiology': [
        'Anticoagulants', 'Antihypertensives', 'Antiarrhythmics'
    ],
    
    # Neurology
    'Neurology': [
        'Anticonvulsants', 'Muscle_Relaxants'
    ],
    
    # Psychiatry
    'Psychiatry': [
        'Antipsychotics', 'Benzodiazepines'
    ],
    
    # Infectious Disease
    'Infectious_Disease': [
        'Antibiotics'
    ],
    
    # Oncology
    'Oncology': [
        'Chemotherapy_Agents', 'Immunosuppressants'
    ],
    
    # Endocrinology
    'Endocrinology': [
        'Antidiabetic_Agents', 'Corticosteroids', 'Vitamin_D_Analogs'
    ],
    
    # Rheumatology
    'Rheumatology': [
        'NSAIDs', 'Corticosteroids'
    ],
    
    # Pain Management
    'Pain_Management': [
        'Opioids', 'NSAIDs'
    ],
    
    # Gastroenterology
    'Gastroenterology': [
        'Proton_Pump_Inhibitors', 'H2_Receptor_Antagonists', 
        'Antiemetics', 'Laxatives', 'GI_Motility_Agents'
    ],
    
    # Pulmonology
    'Pulmonology': [
        'Bronchodilators', 'Antihistamines', 'Corticosteroids'
    ],
    
    # Emergency Medicine
    'Emergency_Medicine': [
        'Anticholinergics', 'Antiemetics'
    ],
    
    # Hematology
    'Hematology': [
        'Anticoagulants', 'Iron_Supplements'
    ],
    
    # Pharmacology (general)
    'Pharmacology': [
        'Other_Drugs'
    ]
}

# ============================================================================
# Disease Categories → Medical Domains
# ============================================================================
print("\n2. Disease_Category → Medical_Domain Mapping")
print("-" * 80)

DISEASE_CATEGORY_TO_DOMAIN = {
    'Cardiology': [
        'Cardiovascular_Diseases'
    ],
    
    'Endocrinology': [
        'Metabolic_Disorders'
    ],
    
    'Pain_Management': [
        'Pain_Disorders'
    ],
    
    'Rheumatology': [
        'Inflammatory_Disorders'
    ],
    
    'Infectious_Disease': [
        'Infectious_Diseases'
    ],
    
    'Neurology': [
        'Neurological_Disorders'
    ],
    
    'Psychiatry': [
        'Psychiatric_Disorders'
    ],
    
    'Pulmonology': [
        'Respiratory_Disorders'
    ],
    
    'Gastroenterology': [
        'Gastrointestinal_Disorders'
    ],
    
    'Oncology': [
        'Oncological_Disorders'
    ],
    
    'General_Medicine': [
        'Fever', 'Other_Conditions'
    ]
}

# ============================================================================
# Toxicity Categories → Medical Domains
# ============================================================================
print("\n3. Toxicity_Category → Medical_Domain Mapping")
print("-" * 80)

TOXICITY_CATEGORY_TO_DOMAIN = {
    'Hepatology': [
        'Organ_Toxicity'  # Includes hepatotoxicity, nephrotoxicity, etc.
    ],
    
    'Oncology': [
        'Reproductive_Toxicity', 'Carcinogenic_Effects'
    ],
    
    'Gastroenterology': [
        'Gastrointestinal_Effects'
    ],
    
    'Cardiology': [
        'Cardiovascular_Effects'
    ],
    
    'Neurology': [
        'Neurological_Effects'
    ],
    
    'Toxicology': [
        'Other_Toxicity'
    ]
}

# ============================================================================
# Create Reverse Mappings
# ============================================================================

# Drug class → domain
drug_class_to_domain = {}
for domain, class_list in DRUG_CLASS_TO_DOMAIN.items():
    for drug_class in class_list:
        if drug_class not in drug_class_to_domain:
            drug_class_to_domain[drug_class] = []
        drug_class_to_domain[drug_class].append(domain)

# Disease category → domain
disease_category_to_domain = {}
for domain, category_list in DISEASE_CATEGORY_TO_DOMAIN.items():
    for category in category_list:
        if category not in disease_category_to_domain:
            disease_category_to_domain[category] = []
        disease_category_to_domain[category].append(domain)

# Toxicity category → domain
toxicity_category_to_domain = {}
for domain, category_list in TOXICITY_CATEGORY_TO_DOMAIN.items():
    for category in category_list:
        if category not in toxicity_category_to_domain:
            toxicity_category_to_domain[category] = []
        toxicity_category_to_domain[category].append(domain)

print("✓ Domain mappings created")

Creating Domain-Level Mappings (φ_CD: Concept → Domain):

1. Drug_Class → Medical_Domain Mapping
--------------------------------------------------------------------------------

2. Disease_Category → Medical_Domain Mapping
--------------------------------------------------------------------------------

3. Toxicity_Category → Medical_Domain Mapping
--------------------------------------------------------------------------------
✓ Domain mappings created


In [14]:
# ## Section 6: Create Domain Mappings DataFrame

In [15]:
print("Creating Domain Mappings DataFrame (φ_CD):")
print("=" * 80)

domain_mappings = []

# Map each concept to its domain(s)
for idx, row in concept_mappings_df.iterrows():
    concept_name = row['concept_name']
    concept_type = row['concept_type']
    
    domains = []
    
    # Determine domain based on concept type
    if concept_type == 'Drug_Class':
        domains = drug_class_to_domain.get(concept_name, ['Pharmacology'])
    elif concept_type == 'Disease_Category':
        domains = disease_category_to_domain.get(concept_name, ['General_Medicine'])
    elif concept_type == 'Toxicity_Category':
        domains = toxicity_category_to_domain.get(concept_name, ['Toxicology'])
    
    # Handle multiple domains (take primary - first one)
    primary_domain = domains[0] if domains else 'General_Medicine'
    
    domain_mappings.append({
        'concept_name': concept_name,
        'concept_type': concept_type,
        'domain_name': primary_domain,
        'mapping_function': 'φ_CD'
    })

domain_mappings_df = pd.DataFrame(domain_mappings)

# Remove duplicates (same concept may appear multiple times from different entities)
domain_mappings_df = domain_mappings_df.drop_duplicates(subset=['concept_name'])

print(f"✓ Created {len(domain_mappings_df)} concept→domain mappings")
print(f"\nDomain distribution:")
print(domain_mappings_df['domain_name'].value_counts())

print(f"\nSample domain mappings:")
print(domain_mappings_df.head(20))

Creating Domain Mappings DataFrame (φ_CD):
✓ Created 39 concept→domain mappings

Domain distribution:
domain_name
Gastroenterology      6
Oncology              5
Neurology             4
Cardiology            4
Endocrinology         3
Psychiatry            3
Pain_Management       2
Rheumatology          2
Pulmonology           2
Infectious_Disease    2
Pharmacology          1
Emergency_Medicine    1
Hematology            1
General_Medicine      1
Toxicology            1
Hepatology            1
Name: count, dtype: int64

Sample domain mappings:
               concept_name concept_type         domain_name mapping_function
0               Other_Drugs   Drug_Class        Pharmacology             φ_CD
1       Antidiabetic_Agents   Drug_Class       Endocrinology             φ_CD
2                   Opioids   Drug_Class     Pain_Management             φ_CD
3           Anticonvulsants   Drug_Class           Neurology             φ_CD
5          Anticholinergics   Drug_Class  Emergency_Medicine 

In [16]:
# ## Section 7: Abstract Entity Relationships to Concept Level

In [17]:
print("Abstracting Entity Relationships to Concept Level:")
print("=" * 80)

def abstract_entity_to_concept(entity_name, concept_mappings_df):
    """Map entity name to concept name"""
    mapping = concept_mappings_df[concept_mappings_df['entity_name'] == entity_name]
    
    if len(mapping) > 0:
        return mapping.iloc[0]['concept_name']
    else:
        return None

# Abstract each entity relationship
print("Processing entity relationships...")
concept_relationships = []

for idx, row in entity_relationships_df.iterrows():
    subject_entity = row['subject']
    predicate = row['predicate']
    object_entity = row['object']
    
    # Map subject and object to concepts
    subject_concept = abstract_entity_to_concept(subject_entity, concept_mappings_df)
    object_concept = abstract_entity_to_concept(object_entity, concept_mappings_df)
    
    # Only create concept relationship if both map successfully
    if subject_concept and object_concept:
        concept_relationships.append({
            'subject_concept': subject_concept,
            'predicate': predicate,
            'object_concept': object_concept,
            'relationship_type': row['relationship_type'],
            'source': 'abstracted_from_entity_level'
        })
    
    # Progress indicator
    if (idx + 1) % 20000 == 0:
        print(f"  Processed {idx+1:,} relationships...")

print(f"✓ Created {len(concept_relationships)} concept-level relationships (before deduplication)")

# Remove duplicates (many entity relationships map to same concept relationship)
concept_relationships_df = pd.DataFrame(concept_relationships)
concept_relationships_df = concept_relationships_df.drop_duplicates(
    subset=['subject_concept', 'predicate', 'object_concept']
)

print(f"✓ Unique concept relationships: {len(concept_relationships_df):,}")

print(f"\nConcept-level predicate distribution:")
print(concept_relationships_df['predicate'].value_counts())

print(f"\nSample concept relationships:")
print(concept_relationships_df.head(20))


Abstracting Entity Relationships to Concept Level:
Processing entity relationships...
  Processed 20,000 relationships...
  Processed 40,000 relationships...
  Processed 60,000 relationships...
  Processed 80,000 relationships...
  Processed 100,000 relationships...
✓ Created 2125 concept-level relationships (before deduplication)
✓ Unique concept relationships: 348

Concept-level predicate distribution:
predicate
interacts_with    225
causes             68
treats             55
Name: count, dtype: int64

Sample concept relationships:
   subject_concept       predicate           object_concept  \
0   Anticoagulants  interacts_with       Immunosuppressants   
1   Anticoagulants  interacts_with                   NSAIDs   
2   Anticoagulants  interacts_with              Other_Drugs   
8   Anticoagulants  interacts_with           Anticoagulants   
11  Anticoagulants  interacts_with      Chemotherapy_Agents   
22     Other_Drugs  interacts_with       Immunosuppressants   
23     Other_Drugs

In [18]:
# ## Section 8: Abstract Concept Relationships to Domain Level

In [19]:
print("Abstracting Concept Relationships to Domain Level (with evidence threshold):")
print("=" * 80)

MIN_CONCEPT_PAIR_SUPPORT = 2  # K=2: require corroboration from 2+ distinct concept pairs
# before promoting a domain-level relationship claim

def abstract_concept_to_domain(concept_name, domain_mappings_df):
    mapping = domain_mappings_df[domain_mappings_df['concept_name'] == concept_name]
    return mapping.iloc[0]['domain_name'] if len(mapping) > 0 else None

# Step 1: map every concept edge to its domain-level (subj, pred, obj) target,
# keeping track of WHICH concept pair supports each domain-level claim
support_records = []
for idx, row in concept_relationships_df.iterrows():
    subject_concept = row['subject_concept']
    predicate = row['predicate']
    object_concept = row['object_concept']

    subject_domain = abstract_concept_to_domain(subject_concept, domain_mappings_df)
    object_domain = abstract_concept_to_domain(object_concept, domain_mappings_df)

    if subject_domain and object_domain:
        support_records.append({
            'subject_domain': subject_domain,
            'predicate': predicate,
            'object_domain': object_domain,
            'supporting_concept_subject': subject_concept,
            'supporting_concept_object': object_concept
        })

support_df = pd.DataFrame(support_records)
print(f"Raw abstracted (domain_subj, predicate, domain_obj) candidates: {len(support_df)}")

# Step 2: count DISTINCT concept-pair support per domain-level claim
support_counts = support_df.groupby(['subject_domain', 'predicate', 'object_domain']).apply(
    lambda g: g[['supporting_concept_subject', 'supporting_concept_object']].drop_duplicates().shape[0]
).reset_index(name='distinct_concept_pair_support')

print(f"Unique domain-level claims before threshold: {len(support_counts)}")
print(f"\nSupport distribution:")
print(support_counts['distinct_concept_pair_support'].value_counts().sort_index())

# Step 3: apply the threshold — only promote claims with enough independent evidence
promoted = support_counts[support_counts['distinct_concept_pair_support'] >= MIN_CONCEPT_PAIR_SUPPORT]
rejected = support_counts[support_counts['distinct_concept_pair_support'] < MIN_CONCEPT_PAIR_SUPPORT]

print(f"\n✓ Promoted to domain_relationships (support >= {MIN_CONCEPT_PAIR_SUPPORT}): {len(promoted)}")
print(f"✗ Rejected — insufficient evidence (support < {MIN_CONCEPT_PAIR_SUPPORT}): {len(rejected)}")
print(f"  These rejected claims are now genuine 'concept says yes, domain says no' cases —")
print(f"  this is what makes scale-inconsistent hallucination testing possible.")

domain_relationships = promoted[['subject_domain', 'predicate', 'object_domain']].copy()
domain_relationships['source'] = f'abstracted_from_concept_level_min_support_{MIN_CONCEPT_PAIR_SUPPORT}'
domain_relationships = domain_relationships.to_dict('records')

# save the rejected set too — needed later for test-fact generation
rejected_domain_claims_df = rejected.merge(
    support_df[['subject_domain', 'predicate', 'object_domain']].drop_duplicates(),
    on=['subject_domain', 'predicate', 'object_domain'], how='left'
)

print(f"\n✓ Created {len(domain_relationships)} concept-level relationships (before dedup, post-threshold)")

domain_relationships_df = pd.DataFrame(domain_relationships)
domain_relationships_df = domain_relationships_df.drop_duplicates(
    subset=['subject_domain', 'predicate', 'object_domain']
)

print(f"✓ Unique concept-derived domain relationships: {len(domain_relationships_df):,}")
print(f"\nDomain-level predicate distribution:")
print(domain_relationships_df['predicate'].value_counts())

Abstracting Concept Relationships to Domain Level (with evidence threshold):
Raw abstracted (domain_subj, predicate, domain_obj) candidates: 348
Unique domain-level claims before threshold: 202

Support distribution:
distinct_concept_pair_support
1    112
2     62
3     14
4      8
5      2
6      2
7      1
9      1
Name: count, dtype: int64

✓ Promoted to domain_relationships (support >= 2): 90
✗ Rejected — insufficient evidence (support < 2): 112
  These rejected claims are now genuine 'concept says yes, domain says no' cases —
  this is what makes scale-inconsistent hallucination testing possible.

✓ Created 90 concept-level relationships (before dedup, post-threshold)
✓ Unique concept-derived domain relationships: 90

Domain-level predicate distribution:
predicate
interacts_with    62
causes            20
treats             8
Name: count, dtype: int64


In [20]:
# ## Section 9: Add Domain-Specific Relationships
# 
# Add the 4 domain-level relationship types we defined

In [21]:
print("Adding Domain-Specific Relationships:")
print("=" * 80)

# Define domain-specific relationships based on medical practice
domain_specific_relationships = [
    # Pharmacology informs clinical specialties
    ('Pharmacology', 'informs', 'Cardiology'),
    ('Pharmacology', 'informs', 'Neurology'),
    ('Pharmacology', 'informs', 'Psychiatry'),
    ('Pharmacology', 'informs', 'Endocrinology'),
    ('Pharmacology', 'informs', 'Rheumatology'),
    ('Pharmacology', 'informs', 'Gastroenterology'),
    ('Pharmacology', 'informs', 'Pulmonology'),
    ('Pharmacology', 'informs', 'Oncology'),
    ('Pharmacology', 'informs', 'Pain_Management'),
    ('Pharmacology', 'informs', 'Infectious_Disease'),
    
    # Clinical collaborations
    ('Cardiology', 'collaborates_with', 'Emergency_Medicine'),
    ('Neurology', 'collaborates_with', 'Emergency_Medicine'),
    ('Psychiatry', 'collaborates_with', 'Neurology'),
    ('Oncology', 'collaborates_with', 'Hematology'),
    ('Rheumatology', 'collaborates_with', 'Pain_Management'),
    ('Pulmonology', 'collaborates_with', 'Cardiology'),
    ('Gastroenterology', 'collaborates_with', 'Hepatology'),
    
    # Clinical specialties use treatments from Pharmacology
    ('Cardiology', 'uses_treatments_from', 'Pharmacology'),
    ('Endocrinology', 'uses_treatments_from', 'Pharmacology'),
    ('Neurology', 'uses_treatments_from', 'Pharmacology'),
    ('Psychiatry', 'uses_treatments_from', 'Pharmacology'),
    ('Oncology', 'uses_treatments_from', 'Pharmacology'),
    ('Rheumatology', 'uses_treatments_from', 'Pharmacology'),
    ('Gastroenterology', 'uses_treatments_from', 'Pharmacology'),
    ('Infectious_Disease', 'uses_treatments_from', 'Pharmacology'),
    
    # Monitoring relationships
    ('Hepatology', 'monitors', 'Pharmacology'),
    ('Toxicology', 'monitors', 'Pharmacology'),
    ('Hematology', 'monitors', 'Cardiology'),
    ('Neurology', 'monitors', 'Pain_Management')
]

# Add to domain relationships
for subj, pred, obj in domain_specific_relationships:
    domain_relationships_df = pd.concat([
        domain_relationships_df,
        pd.DataFrame([{
            'subject_domain': subj,
            'predicate': pred,
            'object_domain': obj,
            'source': 'domain_specific_medical_practice'
        }])
    ], ignore_index=True)

# Remove duplicates again
domain_relationships_df = domain_relationships_df.drop_duplicates(
    subset=['subject_domain', 'predicate', 'object_domain']
)

print(f"✓ Total domain relationships (including domain-specific): {len(domain_relationships_df):,}")

print(f"\nDomain-level predicate distribution:")
print(domain_relationships_df['predicate'].value_counts())

print(f"\nDomain-specific relationships added:")
domain_specific_df = domain_relationships_df[
    domain_relationships_df['source'] == 'domain_specific_medical_practice'
]
print(domain_specific_df.head(20))

Adding Domain-Specific Relationships:
✓ Total domain relationships (including domain-specific): 119

Domain-level predicate distribution:
predicate
interacts_with          62
causes                  20
informs                 10
treats                   8
uses_treatments_from     8
collaborates_with        7
monitors                 4
Name: count, dtype: int64

Domain-specific relationships added:
       subject_domain             predicate       object_domain  \
90       Pharmacology               informs          Cardiology   
91       Pharmacology               informs           Neurology   
92       Pharmacology               informs          Psychiatry   
93       Pharmacology               informs       Endocrinology   
94       Pharmacology               informs        Rheumatology   
95       Pharmacology               informs    Gastroenterology   
96       Pharmacology               informs         Pulmonology   
97       Pharmacology               informs            Oncology

In [22]:
# ## Section 10: Validation & Statistics

In [23]:
print("Validation & Statistics:")
print("=" * 80)

# Entity-level stats
print("\n1. ENTITY LEVEL (G_E)")
print("-" * 80)
print(f"Total entities: {len(entity_catalog_df)}")
print(f"  - Drugs: {len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Drug'])}")
print(f"  - Conditions: {len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Condition'])}")
print(f"  - Adverse Effects: {len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Adverse_Effect'])}")
print(f"\nTotal entity relationships: {len(entity_relationships_df):,}")

# Concept-level stats
print("\n2. CONCEPT LEVEL (G_C)")
print("-" * 80)
unique_concepts = pd.concat([
    concept_mappings_df['concept_name'],
]).unique()
print(f"Total unique concepts: {len(unique_concepts)}")

concept_type_counts = concept_mappings_df['concept_type'].value_counts()
print(f"\nConcept types:")
for ctype, count in concept_type_counts.items():
    print(f"  - {ctype}: {count} mappings")

unique_by_type = concept_mappings_df.groupby('concept_type')['concept_name'].nunique()
print(f"\nUnique concepts by type:")
for ctype, count in unique_by_type.items():
    print(f"  - {ctype}: {count} unique concepts")

print(f"\nTotal concept relationships: {len(concept_relationships_df):,}")

# Domain-level stats
print("\n3. DOMAIN LEVEL (G_D)")
print("-" * 80)
unique_domains = pd.concat([
    domain_mappings_df['domain_name'],
]).unique()
print(f"Total unique domains: {len(unique_domains)}")

print(f"\nDomains:")
for domain in sorted(unique_domains):
    print(f"  - {domain}")

print(f"\nTotal domain relationships: {len(domain_relationships_df):,}")

# Abstraction coverage
print("\n4. ABSTRACTION COVERAGE")
print("-" * 80)
entities_with_concept = concept_mappings_df['entity_name'].nunique()
total_entities = len(entity_catalog_df)
print(f"Entities with concept mapping: {entities_with_concept}/{total_entities} ({entities_with_concept/total_entities*100:.1f}%)")

concepts_with_domain = domain_mappings_df['concept_name'].nunique()
total_concepts = len(unique_concepts)
print(f"Concepts with domain mapping: {concepts_with_domain}/{total_concepts} ({concepts_with_domain/total_concepts*100:.1f}%)")


Validation & Statistics:

1. ENTITY LEVEL (G_E)
--------------------------------------------------------------------------------
Total entities: 149
  - Drugs: 95
  - Conditions: 42
  - Adverse Effects: 12

Total entity relationships: 103,552

2. CONCEPT LEVEL (G_C)
--------------------------------------------------------------------------------
Total unique concepts: 39

Concept types:
  - Drug_Class: 95 mappings
  - Disease_Category: 42 mappings
  - Toxicity_Category: 12 mappings

Unique concepts by type:
  - Disease_Category: 11 unique concepts
  - Drug_Class: 21 unique concepts
  - Toxicity_Category: 7 unique concepts

Total concept relationships: 348

3. DOMAIN LEVEL (G_D)
--------------------------------------------------------------------------------
Total unique domains: 16

Domains:
  - Cardiology
  - Emergency_Medicine
  - Endocrinology
  - Gastroenterology
  - General_Medicine
  - Hematology
  - Hepatology
  - Infectious_Disease
  - Neurology
  - Oncology
  - Pain_Management

In [23]:
# ## Section 11: Generate Outputs

In [24]:
print("Generating Output Files:")
print("=" * 80)

# ============================================================================
# Output 1: Concept Mappings (φ_EC)
# ============================================================================
concept_mappings_df.to_csv(CONCEPT_MAPPINGS_PATH, index=False)
print(f"✓ Saved: {CONCEPT_MAPPINGS_PATH}")
print(f"  Mappings: {len(concept_mappings_df)}")

# ============================================================================
# Output 2: Domain Mappings (φ_CD)
# ============================================================================
domain_mappings_df.to_csv(DOMAIN_MAPPINGS_PATH, index=False)
print(f"\n✓ Saved: {DOMAIN_MAPPINGS_PATH}")
print(f"  Mappings: {len(domain_mappings_df)}")

# ============================================================================
# Output 3: Concept Relationships
# ============================================================================
concept_relationships_df.to_csv(CONCEPT_RELATIONSHIPS_PATH, index=False)
print(f"\n✓ Saved: {CONCEPT_RELATIONSHIPS_PATH}")
print(f"  Relationships: {len(concept_relationships_df):,}")

# ============================================================================
# Output 4: Domain Relationships
# ============================================================================
domain_relationships_df.to_csv(DOMAIN_RELATIONSHIPS_PATH, index=False)
print(f"\n✓ Saved: {DOMAIN_RELATIONSHIPS_PATH}")
print(f"  Relationships: {len(domain_relationships_df):,}")

# ============================================================================
# Output 5: Rejected Relationships
# ============================================================================
REJECTED_DOMAIN_CLAIMS_PATH = INTERMEDIATE_DIR / "rejected_domain_claims.csv"
rejected_domain_claims_df.to_csv(REJECTED_DOMAIN_CLAIMS_PATH, index=False)
print(f"✓ Saved: {REJECTED_DOMAIN_CLAIMS_PATH} ({len(rejected_domain_claims_df)} rejected claims)")

# ============================================================================
# Output 6: Abstraction Report
# ============================================================================
abstraction_report = {
    'generation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    
    'entity_level': {
        'total_entities': int(len(entity_catalog_df)),
        'drugs': int(len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Drug'])),
        'conditions': int(len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Condition'])),
        'adverse_effects': int(len(entity_catalog_df[entity_catalog_df['entity_type'] == 'Adverse_Effect'])),
        'total_relationships': int(len(entity_relationships_df))
    },
    
    'concept_level': {
        'total_concepts': int(len(unique_concepts)),
        'drug_classes': int(unique_by_type.get('Drug_Class', 0)),
        'disease_categories': int(unique_by_type.get('Disease_Category', 0)),
        'toxicity_categories': int(unique_by_type.get('Toxicity_Category', 0)),
        'total_relationships': int(len(concept_relationships_df)),
        'entity_to_concept_mappings': int(len(concept_mappings_df))
    },
    
    'domain_level': {
        'total_domains': int(len(unique_domains)),
        'domains_list': sorted([str(d) for d in unique_domains]),
        'total_relationships': int(len(domain_relationships_df)),
        'concept_to_domain_mappings': int(len(domain_mappings_df))
    },
    
    'abstraction_functions': {
        'phi_EC': {
            'description': 'Entity to Concept mapping',
            'coverage': f"{entities_with_concept}/{total_entities} ({entities_with_concept/total_entities*100:.1f}%)"
        },
        'phi_CD': {
            'description': 'Concept to Domain mapping',
            'coverage': f"{concepts_with_domain}/{total_concepts} ({concepts_with_domain/total_concepts*100:.1f}%)"
        },
        'phi_ED': {
            'description': 'Entity to Domain (transitive)',
            'note': 'φ_ED = φ_CD ∘ φ_EC'
        }
    },
    
    'top_concepts': {
        'drug_classes': concept_mappings_df[
            concept_mappings_df['concept_type'] == 'Drug_Class'
        ]['concept_name'].value_counts().head(10).to_dict(),
        'disease_categories': concept_mappings_df[
            concept_mappings_df['concept_type'] == 'Disease_Category'
        ]['concept_name'].value_counts().head(10).to_dict(),
        'toxicity_categories': concept_mappings_df[
            concept_mappings_df['concept_type'] == 'Toxicity_Category'
        ]['concept_name'].value_counts().head(10).to_dict()
    },
    
    'relationship_abstraction': {
        'entity_relationships': int(len(entity_relationships_df)),
        'concept_relationships': int(len(concept_relationships_df)),
        'domain_relationships': int(len(domain_relationships_df)),
        'abstraction_ratio': {
            'entity_to_concept': f"1:{len(entity_relationships_df)/len(concept_relationships_df):.1f}",
            'concept_to_domain': f"1:{len(concept_relationships_df)/len(domain_relationships_df):.1f}"
        }
    }
}

with open(ABSTRACTION_REPORT_PATH, 'w') as f:
    json.dump(abstraction_report, f, indent=2)

print(f"\n✓ Saved: {ABSTRACTION_REPORT_PATH}")

Generating Output Files:
✓ Saved: ../Datasets/intermediate/concept_mappings.csv
  Mappings: 149

✓ Saved: ../Datasets/intermediate/domain_mappings.csv
  Mappings: 39

✓ Saved: ../Datasets/intermediate/concept_relationships.csv
  Relationships: 348

✓ Saved: ../Datasets/intermediate/domain_relationships.csv
  Relationships: 119
✓ Saved: ../Datasets/intermediate/rejected_domain_claims.csv (112 rejected claims)

✓ Saved: ../Datasets/intermediate/abstraction_report.json


In [25]:
# ## Section 12: Final Summary

In [26]:
print("\n" + "=" * 80)
print("COMPONENT 3 COMPLETE - FINAL SUMMARY")
print("=" * 80)

print(f"\n✓ Multi-Scale Knowledge Graph Hierarchy Created:")

print(f"\n  ENTITY LEVEL (G_E):")
print(f"    - Entities: {len(entity_catalog_df)}")
print(f"    - Relationships: {len(entity_relationships_df):,}")

print(f"\n  CONCEPT LEVEL (G_C):")
print(f"    - Concepts: {len(unique_concepts)}")
print(f"    - Relationships: {len(concept_relationships_df):,}")
print(f"    - φ_EC mappings: {len(concept_mappings_df)}")

print(f"\n  DOMAIN LEVEL (G_D):")
print(f"    - Domains: {len(unique_domains)}")
print(f"    - Relationships: {len(domain_relationships_df):,}")
print(f"    - φ_CD mappings: {len(domain_mappings_df)}")

print(f"\n✓ Abstraction Functions:")
print(f"    - φ_EC: Entity → Concept ({entities_with_concept}/{total_entities} coverage)")
print(f"    - φ_CD: Concept → Domain ({concepts_with_domain}/{total_concepts} coverage)")
print(f"    - φ_ED: Entity → Domain (transitive)")

print(f"\n✓ Files Generated:")
print(f"    1. {CONCEPT_MAPPINGS_PATH.name}")
print(f"    2. {DOMAIN_MAPPINGS_PATH.name}")
print(f"    3. {CONCEPT_RELATIONSHIPS_PATH.name}")
print(f"    4. {DOMAIN_RELATIONSHIPS_PATH.name}")
print(f"    5. {ABSTRACTION_REPORT_PATH.name}")

print(f"\n✓ Next Steps:")
print(f"    1. Review abstraction quality")
print(f"    2. Validate concept and domain mappings")
print(f"    3. Check relationship abstraction ratios")
print(f"    4. Proceed to Component 4: Knowledge Graph Construction")

print("\n" + "=" * 80)
print("END OF COMPONENT 3")
print("=" * 80)


COMPONENT 3 COMPLETE - FINAL SUMMARY

✓ Multi-Scale Knowledge Graph Hierarchy Created:

  ENTITY LEVEL (G_E):
    - Entities: 149
    - Relationships: 103,552

  CONCEPT LEVEL (G_C):
    - Concepts: 39
    - Relationships: 348
    - φ_EC mappings: 149

  DOMAIN LEVEL (G_D):
    - Domains: 16
    - Relationships: 231
    - φ_CD mappings: 39

✓ Abstraction Functions:
    - φ_EC: Entity → Concept (149/149 coverage)
    - φ_CD: Concept → Domain (39/39 coverage)
    - φ_ED: Entity → Domain (transitive)

✓ Files Generated:
    1. concept_mappings.csv
    2. domain_mappings.csv
    3. concept_relationships.csv
    4. domain_relationships.csv
    5. abstraction_report.json

✓ Next Steps:
    1. Review abstraction quality
    2. Validate concept and domain mappings
    3. Check relationship abstraction ratios
    4. Proceed to Component 4: Knowledge Graph Construction

END OF COMPONENT 3


## Testing Newly generated rejected claims

In [25]:
import pandas as pd
rejected = pd.read_csv("../Datasets/intermediate/rejected_domain_claims.csv")
print(rejected.shape)
print(rejected.head(10))
print(rejected['predicate'].value_counts())

(112, 4)
       subject_domain       predicate  object_domain  \
0          Cardiology          causes      Neurology   
1          Cardiology  interacts_with  Endocrinology   
2          Cardiology  interacts_with     Hematology   
3          Cardiology  interacts_with    Pulmonology   
4          Cardiology          treats  Endocrinology   
5  Emergency_Medicine          causes      Neurology   
6  Emergency_Medicine  interacts_with      Neurology   
7  Emergency_Medicine  interacts_with   Pharmacology   
8  Emergency_Medicine  interacts_with     Psychiatry   
9  Emergency_Medicine  interacts_with    Pulmonology   

   distinct_concept_pair_support  
0                              1  
1                              1  
2                              1  
3                              1  
4                              1  
5                              1  
6                              1  
7                              1  
8                              1  
9                       

In [26]:
import pandas as pd
domain_rel = pd.read_csv("../Datasets/intermediate/domain_relationships.csv")
print(f"Total domain_relationships.csv rows: {len(domain_rel)}")
print(f"\nBy source:")
print(domain_rel['source'].value_counts())
print(f"\nPredicate distribution:")
print(domain_rel['predicate'].value_counts())

Total domain_relationships.csv rows: 119

By source:
source
abstracted_from_concept_level_min_support_2    90
domain_specific_medical_practice               29
Name: count, dtype: int64

Predicate distribution:
predicate
interacts_with          62
causes                  20
informs                 10
treats                   8
uses_treatments_from     8
collaborates_with        7
monitors                 4
Name: count, dtype: int64
